# Phase 2 & 3 — Multi-Source Data Collection & Automated ETL

**CRISP-DM Stages:** Data Understanding & Data Preparation  
**Sources Integrated:**
1. Relational SQL Database (`enterprise_ops.db` / PostgreSQL schema)
2. External REST APIs (Open-ER FX Rates & World Bank GDP Growth)
3. Web Scraper (`BeautifulSoup4` competitive pricing benchmarks)


In [1]:
import json
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.config import settings
from src.ingestion.sql_connector import SQLConnector
from src.ingestion.api_connector import MacroAPIConnector
from src.ingestion.web_scraper import CompetitorPriceScraper

sql_conn = SQLConnector()
raw_tables = sql_conn.extract_all_raw()
for tbl, df in raw_tables.items():
    print(f'Raw SQL Table {tbl:28s}: {df.shape}')


2026-11-20 16:39:40 | INFO     | src.ingestion.sql_connector | Extracting table 'dim_regions' from SQL database...


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracted 12 rows from 'dim_regions'.


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracting table 'dim_products' from SQL database...


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracted 60 rows from 'dim_products'.


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracting table 'dim_customers' from SQL database...


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracted 4000 rows from 'dim_customers'.


... (3 outputs left out)

2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracted 12338 rows from 'fact_customer_interactions'.


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracting table 'customer_future_outcomes' from SQL database...


2026-11-20 16:39:41 | INFO     | src.ingestion.sql_connector | Extracted 4000 rows from 'customer_future_outcomes'.


Raw SQL Table dim_regions                 : (12, 9)
Raw SQL Table dim_products                : (60, 8)
Raw SQL Table dim_customers               : (4000, 10)
Raw SQL Table fact_orders                 : (36554, 15)
Raw SQL Table fact_customer_interactions  : (12338, 10)
Raw SQL Table customer_future_outcomes    : (4000, 4)


In [2]:
macro_df = MacroAPIConnector().fetch_macro_indicators()
comp_df = CompetitorPriceScraper().scrape_competitor_benchmarks()
display(macro_df.head(6))
display(comp_df)


2026-11-20 16:39:41 | INFO     | src.ingestion.api_connector | Loaded macro indicators from SQLite cache.


2026-11-20 16:39:41 | INFO     | src.ingestion.web_scraper | Loaded scraped competitor benchmarks from SQLite cache.


,country_code,currency_code,fx_rate_to_usd,gdp_growth_pct,indicator_year,source_provider,ingested_at_utc
0,USA,USD,1.0000,2.16,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00
1,CAN,CAD,1.4325,1.74,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00
2,GBR,GBP,0.7624,1.39,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00
3,DEU,EUR,0.9107,0.24,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00
4,FRA,EUR,0.9107,0.84,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00
5,EGY,EGP,53.8146,4.39,2025,Open-ER API + World Bank API,2026-11-20T14:39:41.769382+00:00


,category,competitor_avg_price,competitor_price_index,market_rating_avg,in_stock_rate,scraped_sample_count,source_url,ingested_at_utc
0,Cloud Software,415.38,0.989,3.50,1.0,4,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00
1,Enterprise Hardware,735.76,1.082,2.75,1.0,4,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00
2,Analytics Modules,322.32,0.948,3.33,1.0,3,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00
3,IoT Sensors,222.50,1.141,2.67,1.0,3,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00
4,Security Suites,468.44,0.956,2.67,1.0,3,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00
5,Managed Services,554.40,0.990,2.00,1.0,3,https://books.toscrape.com/,2026-11-20T14:31:24.989343+00:00


In [3]:
dq = json.loads((settings.processed_dir / 'data_quality_report.json').read_text())
print('=== ETL Data Quality Audit Report ===')
for k, v in dq.items():
    print(f'{k:36s}: {v}')


=== ETL Data Quality Audit Report ===
raw_orders_count                    : 36554
clean_orders_count                  : 36014
duplicate_orders_removed            : 540
raw_interactions_count              : 12338
clean_interactions_count            : 12156
duplicate_interactions_removed      : 182
raw_customers_count                 : 4000
clean_customers_count               : 4000
missing_values_imputed              : 3148
outliers_clamped                    : 240
casing_standardized_count           : 1582
validation_records_checked          : 52188
validation_errors_count             : 0
post_cleaning_error_rate_pct        : 0.0
pipeline_duration_seconds           : 11.767


In [4]:
c360 = pd.read_csv(settings.processed_dir / 'customer_360.csv')
print(f'Unified Customer 360 Feature Matrix: {c360.shape}')
print('Missing values in Customer 360:', int(c360.isna().sum().sum()))
display(c360.head())


Unified Customer 360 Feature Matrix: (4000, 35)


Missing values in Customer 360: 0


,customer_id,customer_code,segment,contract_type,acquisition_channel,account_tenure_days,macro_region,country_code,country_name,order_frequency,...,fx_rate_to_usd,is_churned,next_12m_revenue_usd,competitor_price_index,spend_momentum_ratio,log_hist_revenue,cluster_id,cluster_label,pca_1,pca_2
0,1,CUST-00001,SMB,Annual,Inbound Web,543,Europe,DEU,Germany,4,...,0.9107,1,767.58,1.082,0.0000,8.1719,0,Dormant Price-Sensitive,-1.0952,-1.4374
1,2,CUST-00002,SMB,Annual,Direct Sales,696,North America,USA,United States,4,...,1.0000,0,579.72,0.956,0.0000,8.0841,0,Dormant Price-Sensitive,-1.3194,-1.0102
2,3,CUST-00003,Mid-Market,2-Year,Inbound Web,495,Europe,FRA,France,10,...,0.9107,0,10529.38,0.956,1.0032,9.2504,1,Loyal Growth Mid-Market,1.1842,-0.0450
3,4,CUST-00004,Mid-Market,Annual,Direct Sales,628,APAC & LATAM,JPN,Japan,9,...,159.7495,0,9905.87,1.082,1.3977,9.6744,1,Loyal Growth Mid-Market,1.0483,0.1932
4,5,CUST-00005,Mid-Market,Annual,Direct Sales,670,APAC & LATAM,AUS,Australia,3,...,1.4987,1,97.55,0.989,0.0000,7.1168,0,Dormant Price-Sensitive,-2.4679,-0.7156


### Key Preparation Takeaways
- Duplicate business keys (`order_code`, `ticket_code`), whitespace/casing inconsistencies, numeric outliers (`delivery_days = 999`, negative shipping), and missingness were cleanly remediated.
- Post-ETL Pydantic v2 validation recorded **0.00% schema error rate** (< 0.1%).
- Historical features are anchored at `2025-12-31` to eliminate data leakage.